# NB04: False-Alarm Gate (Stage A + Stage B)

Fit Stage A logistic regression on **DS1 train**; calibrate thresholds on **DS1 val** only. DS2/fold 10 still untouched.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab — skip Drive mount.')

## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
import json
from pathlib import Path


def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )


REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner, ensure_dir, student_variants_for_run, student_checkpoint_path, variant_result_path
DRY_RUN = resolve_dry_run()
import torch
from src.config import (
    MODEL_DIR, COMMON_FS, MITBIH_NORMAL_CLASS, SENSITIVITY_LOSS_BUDGET,
    RESULTS_DIR, GATE_MIN_FALSE_ALARMS_FOR_LR, MITBIH_DS1_VAL, AAMI_CLASS_TO_IDX,
    STUDENT_TRUNK_CHANNELS,
)
from src.data.mitbih import MITBIHDataset
from src.explain.gradcam import grad_cam_1d
from src.gate.stage_a import stage_a_score, decide_stage_a
from src.gate.stage_b import stage_b_decision
from src.gate.calibration import fit_gate_model, calibrate_thresholds, calibrate_baseline_thresholds, fallback_risk
from src.gate.evaluation import collect_alarm_records, records_to_tuples
from src.models.student import MultiTaskStudent
from src.models.teacher_a import TeacherA

device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')
ensure_dir(RESULTS_DIR)

STUDENT_VARIANTS_TO_RUN = student_variants_for_run(DRY_RUN)
print_run_banner(notebook='NB04_Gate', dry_run=DRY_RUN, variants=STUDENT_VARIANTS_TO_RUN)

In [ ]:
for VARIANT in STUDENT_VARIANTS_TO_RUN:
    print('\n' + '=' * 60 + f'\nStudent variant: {VARIANT}\n' + '=' * 60)
    STUDENT_PATH = student_checkpoint_path(VARIANT)
    print(f'Checkpoint: {STUDENT_PATH}')
    student = MultiTaskStudent(c_internal=STUDENT_TRUNK_CHANNELS).to(device)
    student.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
    teacher_a = TeacherA().to(device)
    teacher_a.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'teacher_a.pt'), map_location=device))
    student.eval(); teacher_a.eval()

    record_ids = MITBIH_DS1_VAL[:3] if DRY_RUN else MITBIH_DS1_VAL
    mit_val = MITBIHDataset(record_ids, dry_run=DRY_RUN, use_cache=True)
    normal_idx = AAMI_CLASS_TO_IDX[MITBIH_NORMAL_CLASS]
    print(f'Validation beats: {len(mit_val)} (dry_run={DRY_RUN})')

    val_records, _ = collect_alarm_records(
        mit_val,
        student,
        normal_idx=normal_idx,
        fs=COMMON_FS,
        device=device,
        student_layer=MultiTaskStudent.LAST_CONV,
        max_alarms=int(os.environ.get('XFAD_GATE_MAX_ALARMS', '0')),
        progress=not DRY_RUN,
    )
    alarms = records_to_tuples(val_records)
    n_false_alarms = sum(1 for *_, fa in alarms if fa)
    print(f'Validation alarm events: {len(alarms)} '
          f'({n_false_alarms} false alarms, {len(alarms) - n_false_alarms} true alarms)')

    if DRY_RUN:
        while len(alarms) < 60:
            alarms.append((0.2, 0.3, 0.6, True))
            alarms.append((0.8, 0.9, 0.95, False))

    gate_model = fit_gate_model(alarms)
    lr_model_fitted = gate_model is not None
    print('Stage-A scorer:', 'logistic regression' if lr_model_fitted
          else f'fallback rule (needs {GATE_MIN_FALSE_ALARMS_FOR_LR}+ false alarms)')

    thresholds = calibrate_thresholds(alarms, gate_model)
    baseline_thresholds = calibrate_baseline_thresholds(alarms, budget=SENSITIVITY_LOSS_BUDGET)
    print(f'Thresholds: low={thresholds.low:.4f} high={thresholds.high:.4f}')
    print('Budget-matched baselines (validation):')
    for name, bt in baseline_thresholds.items():
        print(f'  {name}: threshold={bt.threshold:.4f}, FA reduction {bt.calibration_fa_reduction * 100:.1f}% '
              f'at sensitivity loss {bt.calibration_sensitivity_loss * 100:.2f}%')
    print(f'Validation FA reduction {thresholds.calibration_fa_reduction * 100:.1f}% at '
          f'sensitivity loss {thresholds.calibration_sensitivity_loss * 100:.2f}% '
          f'(budget {SENSITIVITY_LOSS_BUDGET * 100:.0f}%)')
    print(f'Stage-B escalation rate: {thresholds.escalation_rate * 100:.1f}% of alarms')
    if thresholds.used_fallback:
        print('WARNING: no threshold achieved any FA reduction within budget - report as a negative result')

    qo0, sqi0, pc0, _ = alarms[0]
    risk0 = stage_a_score(gate_model, qo0, sqi0, pc0) if lr_model_fitted else fallback_risk(qo0, sqi0, pc0)
    print('Stage A demo:', risk0, decide_stage_a(risk0, thresholds.low, thresholds.high))

    xb = mit_val[0][0].unsqueeze(0).to(device)
    s_pred = int(student(xb, 'A').argmax(1).item())
    t_pred = int(teacher_a(xb).argmax(1).item())
    s_cam = grad_cam_1d(student, xb, s_pred, MultiTaskStudent.LAST_CONV, task='A')[0]
    t_cam = grad_cam_1d(teacher_a, xb, t_pred, TeacherA.LAST_CONV)[0]

    assert stage_b_decision(s_pred, s_pred, s_cam, t_cam, normal_idx, connectivity_available=True) == 'TRIGGER'

    abnormal_idx = next(i for i in range(5) if i != normal_idx)
    seq_len = s_cam.shape[-1]

    low_div_s = torch.zeros(seq_len)
    low_div_t = torch.zeros(seq_len)
    low_div_s[100:150] = 1.0
    low_div_t[105:145] = 1.0
    assert stage_b_decision(
        abnormal_idx, normal_idx, low_div_s, low_div_t, normal_idx, connectivity_available=True,
    ) == 'SUPPRESS'

    high_div_s = torch.zeros(seq_len)
    high_div_t = torch.zeros(seq_len)
    high_div_s[50:100] = 1.0
    high_div_t[150:200] = 1.0
    assert stage_b_decision(
        abnormal_idx, normal_idx, high_div_s, high_div_t, normal_idx, connectivity_available=True,
    ) == 'SUPPRESS_AND_LOG_FOR_REVIEW'

    assert stage_b_decision(s_pred, s_pred, s_cam, t_cam, normal_idx, connectivity_available=False) == 'TRIGGER'
    print(f'Beat 0 demo preds: student={s_pred}, teacher={t_pred}')
    print('Stage B assertions passed')

    artifact = {
        'low_threshold': thresholds.low,
        'high_threshold': thresholds.high,
        'divergence_threshold': thresholds.divergence,
        'sensitivity_budget': SENSITIVITY_LOSS_BUDGET,
        'used_fallback': thresholds.used_fallback,
        'lr_model_fitted': lr_model_fitted,
        'lr_min_false_alarms': GATE_MIN_FALSE_ALARMS_FOR_LR,
        'student_variant': VARIANT,
        'baseline_calibration': {
            name: {
                'threshold': bt.threshold,
                'fa_reduction': bt.calibration_fa_reduction,
                'sensitivity_loss': bt.calibration_sensitivity_loss,
            }
            for name, bt in baseline_thresholds.items()
        },
        'calibration': {
            'dry_run': DRY_RUN,
            'population': 'student alarm events on MIT-BIH DS1 validation records',
            'records': [int(r) for r in record_ids],
            'n_alarm_events': len(alarms),
            'n_false_alarms': n_false_alarms,
            'fa_reduction': thresholds.calibration_fa_reduction,
            'sensitivity_loss': thresholds.calibration_sensitivity_loss,
            'escalation_rate': thresholds.escalation_rate,
        },
    }
    with open(variant_result_path('gate_thresholds', VARIANT), 'w') as f:
        json.dump(artifact, f, indent=2)
    if gate_model is not None:
        import joblib
        gate_model_path = variant_result_path('gate_model', VARIANT, ext='.joblib')
        joblib.dump(gate_model, gate_model_path)
        print('Saved', gate_model_path, '(learned Stage-A scorer)')
    else:
        print('No gate_model saved: Stage A uses the fallback rule - state this in the paper')
    print('Gate artifacts saved to', RESULTS_DIR)
